In [1]:
! pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.5 MB/s eta 0:00:00


In [2]:
! apt-get install -y seqkit

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  seqkit
0 upgraded, 1 newly installed, 0 to remove and 6 not upgraded.
Need to get 6,544 kB of archives.
After this operation, 15.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 seqkit amd64 2.1.0+ds-1ubuntu0.1 [6,544 kB]
Fetched 6,544 kB in 2s (2,865 kB/s)
Selecting previously unselected package seqkit.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../seqkit_2.1.0+ds-1ubuntu0.1_amd64.deb ...
Unpacking seqkit (2.1.0+ds-1ubuntu0.1) ...
Setting up seqkit (2.1.0+ds-1ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
import requests
import re
import json
from Bio import SeqIO
import subprocess
import sys


class MyFastaParser:
    def __init__(self, file_name):
        self.filename = file_name


    def _get_uniprot(self, accession):
        endpoint = "https://rest.uniprot.org/uniprotkb/accessions"
        return requests.get(endpoint, params={'accessions': accession})

    def _get_ensembl(self, gene_id):
        endpoint = f"https://rest.ensembl.org/lookup/id/{gene_id}"
        return requests.get(endpoint, headers={"Content-Type": "application/json"})


    def _uniprot_parse_response(self, resp):
        data = resp.json()
        results = data.get("results", [])

        output = {}
        for val in results:
            acc = val.get('primaryAccession')
            species = val.get('organism', {}).get('scientificName')
            gene = val.get('genes')
            seq = val.get('sequence')

            output[acc] = {
                'organism': species,
                'geneInfo': gene,
                'sequenceInfo': seq,
                'type': 'protein'
            }
        return output

    def _ensembl_parse_response(self, resp):
        data = resp.json()

        if "error" in data:
            return {"error": data["error"]}

        return {
            data.get("id"): {
                "object_type": data.get("object_type"),
                "species": data.get("species"),
                "assembly_name": data.get("assembly_name"),
                "biotype": data.get("biotype"),
                "display_name": data.get("display_name"),
                "description": data.get("description"),
                "type": "nucleotide"
            }
        }


    def _access_database(self, id, database, seq_description, seq_sequence):

        result = {
            "description": seq_description,
            "sequence": str(seq_sequence),
            "database": database
        }

        try:
            if database == "uniprot":
                resp = self._get_uniprot(id)
                if resp.status_code == 200:
                    parsed = self._uniprot_parse_response(resp)
                    result["db_info"] = parsed.get(id, {})
                else:
                    result["db_info"] = f"error:{resp.status_code}"

            elif database == "ensembl":
                resp = self._get_ensembl(id)
                if resp.status_code == 200:
                    parsed = self._ensembl_parse_response(resp)
                    result["db_info"] = parsed.get(id, {})
                else:
                    result["db_info"] = f"error:{resp.status_code}"

        except Exception as e:
            result["db_info"] = str(e)

        return result


    def seqkit_stats(self):

      try:
          proc = subprocess.run(
              ["seqkit", "stats", "-a", "-T", self.filename],
              capture_output=True,
              text=True
          )

          if proc.returncode != 0:
              return {"error": proc.stderr.strip()}

          lines = proc.stdout.strip().split("\n")

          header = lines[0].split("\t")
          values = lines[1].split("\t")

          stats = dict(zip(header, values))

          file_type = "DNA"
          if "protein" in proc.stdout.lower():
              file_type = "Protein"

          stats["type"] = file_type

          return {
              "fasta_seqkit_stat_info": stats,
              "fasta_type": file_type,
              "fasta_num_seqs": int(stats.get("num_seqs", 0))
          }

      except Exception as e:
          return {"error": str(e)}


    def biopython_parser(self, seqkit_result):

        if "error" in seqkit_result:
            return seqkit_result

        output = {}

        uniprot_pattern = r"(?:sp|tr)\|([A-Z0-9]{6})\|"
        ensembl_pattern = r"(ENS[A-Z]*G\d+)"

        file_type = seqkit_result.get("fasta_type")

        db_name = None

        for record in SeqIO.parse(self.filename, "fasta"):

            desc = record.description
            seq = record.seq

            found_id = None
            db = None

            if file_type == "Protein":
                match = re.search(uniprot_pattern, desc)
                if match:
                    found_id = match.group(1)
                    db = "uniprot"

            elif file_type == "DNA":
                match = re.search(ensembl_pattern, desc)
                if match:
                    found_id = match.group(1)
                    db = "ensembl"

            if found_id:
                db_name = db

                result = self._access_database(found_id, db, desc, seq)

                output[f"file_info_{found_id}"] = {
                    "description": desc,
                    "sequence": str(seq)
                }

                output[f"database_info_{found_id}"] = result.get("db_info", {})

            else:
                output["WARNING"] = {"No ID match found."}

        if db_name:
            output = {"DB_name": db_name, **output}

        return output


    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [4]:
parser = MyFastaParser('/content/test_file.fasta')
stats = parser.seqkit_stats()
print(stats)

{'fasta_seqkit_stat_info': {'file': '/content/test_file.fasta', 'format': 'FASTA', 'type': 'Protein', 'num_seqs': '2', 'sum_len': '456', 'min_len': '29', 'avg_len': '228.0', 'max_len': '427', 'Q1': '29.0', 'Q2': '228.0', 'Q3': '427.0', 'sum_gap': '0', 'N50': '427', 'Q20(%)': '0.00', 'Q30(%)': '0.00'}, 'fasta_type': 'Protein', 'fasta_num_seqs': 2}


In [5]:
biopython = parser.biopython_parser(stats)

In [6]:
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS